# Feature Engineering
- **Dataset**: Crop Recommendation Dataset
- **Source**: Kaggle — Crop Recommendation Dataset
- **Notebook**: 02 — Feature Engineering
- **Authors**: Group E




## 1.Scope

We keep all original features from the dataset:

- `N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`

We add only 3 engineered features with agronomic motivation, then test if they improve classification performance.



In [33]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, mutual_info_classif

SEED = 42
pd.set_option("display.max_columns", 100)


In [34]:
DATA_PATH = "../data/raw/Crop_recommendation.csv"
df = pd.read_csv(DATA_PATH)

BASE_FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"

print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Classes:", df[TARGET].nunique())


Shape: (2200, 8)
Missing values: 0
Duplicate rows: 0
Classes: 22


## 2.Feature Engineering Hypotheses 
We selected these engineered features to add meaningful complexity based on agronomic reasoning, not random transformations. `moisture_index` combines rainfall and humidity to better represent effective water conditions, `P_to_K` captures nutrient balance between phosphorus and potassium rather than only absolute values, and `ph_dev` models how far soil pH is from a near-optimal zone for many crops. Together, these features test whether simple domain-informed interactions and nonlinear effects improve crop classification while still keeping all original dataset features.

### H1: Moisture Availability Index (log-transformed)

$$
\text{moisture\_index} = \log(1 + (\text{rainfall} \times \text{humidity}))
$$

Rainfall and humidity jointly describe water availability.  
Because the product is right-skewed, we apply `log1p` to reduce skewness and stabilize scale for Logistic Regression.


### H2: Nutrient Balance Ratio

$$
\text{P\_to\_K} = \frac{P}{K}
$$

Crop response depends on nutrient balance, not only absolute nutrient amounts.

### H3: pH Deviation from Near-Optimal Zone

$$
\text{ph\_dev} = |ph - 6.5|
$$

Many crops prefer slightly acidic to neutral soil. Distance from this zone can capture nonlinear effects.



In [35]:
def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = 1e-6

    X["moisture_index"] = np.log1p(X["rainfall"] * X["humidity"])
    X["P_to_K"] = X["P"] / (X["K"] + eps)
    X["ph_dev"] = np.abs(X["ph"] - 6.5)

    return X


## 3.Data Quality Checks for New Features



In [36]:
X_base = df[BASE_FEATURES].copy()
y = df[TARGET].copy()

X_all = add_engineered_features(X_base)

ENGINEERED_FEATURES = ["moisture_index", "P_to_K", "ph_dev"]
FINAL_FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

print("Base features:", len(BASE_FEATURES))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("Total features:", len(FINAL_FEATURES))

X_all[ENGINEERED_FEATURES].describe().T


Base features: 7
Engineered features: 3
Total features: 10


,count,mean,std,min,25%,50%,75%,max
moisture_index,2200.0,8.686914,0.716235,6.846276,8.240610,8.666980,9.214333,10.139467
P_to_K,2200.0,1.668555,1.197217,0.090909,0.697874,1.262531,2.588235,5.999999
ph_dev,2200.0,0.589235,0.502549,0.000064,0.227128,0.475365,0.813645,3.435091


Checks performed:
- No missing values introduced
- No infinite values from division
- Feature ranges are plausible


## 3.1. Engineered Feature Correlations 

In [37]:
print("NaN in engineered:", X_all[ENGINEERED_FEATURES].isna().any().any())
print("Inf in engineered:", np.isinf(X_all[ENGINEERED_FEATURES].to_numpy()).any())

X_all[ENGINEERED_FEATURES].corr()


NaN in engineered: False
Inf in engineered: False


,moisture_index,P_to_K,ph_dev
moisture_index,1.000000,-0.270885,-0.187665
P_to_K,-0.270885,1.000000,0.193807
ph_dev,-0.187665,0.193807,1.000000


## 4. Ablation Study: Measuring the Contribution of Each Engineered Feature

To quantify the value of each engineered variable, we evaluate models in a stepwise manner: first with the 7 original features (baseline), then by adding engineered features one at a time, and finally with all engineered features together. This isolates the performance impact of each addition.

We use **Logistic Regression** as a  baseline classifier and report **Macro-F1** under **Stratified 5-Fold Cross-Validation** to ensure robust, class-balanced evaluation.


In [38]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(X: pd.DataFrame, y: pd.Series):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=8000)),
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1_macro")
    return scores.mean(), scores.std()

experiments = {
    "base_7_only": BASE_FEATURES,
    "base + moisture_index": BASE_FEATURES + ["moisture_index"],
    "base + P_to_K": BASE_FEATURES + ["P_to_K"],
    "base + ph_dev": BASE_FEATURES + ["ph_dev"],
    "base + all_3_engineered": FINAL_FEATURES,
}

rows = []
for name, cols in experiments.items():
    mean_f1, std_f1 = evaluate(X_all[cols], y)
    rows.append({
        "experiment": name,
        "n_features": len(cols),
        "macro_f1_mean": round(mean_f1, 4),
        "macro_f1_std": round(std_f1, 4),
    })

ablation_df = pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False)
ablation_df


,experiment,n_features,macro_f1_mean,macro_f1_std
4,base + all_3_engineered,10,0.9795,0.0058
3,base + ph_dev,8,0.9750,0.0039
1,base + moisture_index,8,0.9725,0.0033
2,base + P_to_K,8,0.9713,0.0052
0,base_7_only,7,0.9707,0.0023


### Interpretation of Ablation Results

The baseline model using the 7 original features achieves a Macro-F1 of **0.9707**.  
All engineered-feature variants improve over baseline:

- `moisture_index` (log-transformed): **0.9725** (**+0.0018**)
- `P_to_K`: **0.9713** (**+0.0006**)
- `ph_dev`: **0.9750** (**+0.0043**)
- all 3 engineered features: **0.9795** (**+0.0088**)

Key observations:
1. **`ph_dev` is the strongest single engineered feature**, indicating that nonlinear pH effects are important for class separation.
2. **`moisture_index` now contributes more clearly** after log transformation, consistent with reduced skewness and improved scaling behavior.
3. **The combined model performs best**, suggesting the three engineered features provide complementary information.




In [39]:
fs_rows = []
for k in [7, 8, 9, 10]:
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=mutual_info_classif, k=k)),
        ("model", LogisticRegression(max_iter=8000)),
    ])
    scores = cross_val_score(pipe, X_all[FINAL_FEATURES], y, cv=cv, scoring="f1_macro")
    fs_rows.append({
        "k_selected": k,
        "macro_f1_mean": round(scores.mean(), 4),
        "macro_f1_std": round(scores.std(), 4),
    })

pd.DataFrame(fs_rows)


,k_selected,macro_f1_mean,macro_f1_std
0,7,0.9647,0.0135
1,8,0.9739,0.0069
2,9,0.9721,0.0045
3,10,0.9795,0.0058


### 5.1. Features Selected for Each k

Below we list the exact features selected when `k` is varied from 7 to 10.


In [40]:
selected_rows = []

for k in [7, 8, 9, 10]:
    selector_k = SelectKBest(score_func=mutual_info_classif, k=k)
    selector_k.fit(X_fs, y)

    selected_features = X_fs.columns[selector_k.get_support()].tolist()

    selected_rows.append({
        "k_selected": k,
        "selected_features": ", ".join(selected_features)
    })

pd.DataFrame(selected_rows)


,k_selected,selected_features
0,7,"P, K, temperature, humidity, rainfall, moistur..."
1,8,"N, P, K, temperature, humidity, rainfall, mois..."
2,9,"N, P, K, temperature, humidity, ph, rainfall, ..."
3,10,"N, P, K, temperature, humidity, ph, rainfall, ..."


In [ ]:
# Initialize and train the Logistic Regression model
# max_iter is increased because multi-class problems might require more iterations to converge
log_reg_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)
log_reg_model.fit(X_train_scaled, y_train)

log_reg_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
log_reg_model.fit(X_train_scaled, y_train)

TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

### 5.2. Ranked Feature Importance (Mutual Information)

To make feature selection interpretable, we rank all candidate features (`k='all'`) using the same score function (`mutual_info_classif`) used in SelectKBest.

Note: this ranking is for interpretation only; model performance is still evaluated with cross-validation.


In [41]:
# Rank all features by mutual information score
X_fs = X_all[FINAL_FEATURES]

selector_all = SelectKBest(score_func=mutual_info_classif, k="all")
selector_all.fit(X_fs, y)

rank_df = pd.DataFrame({
    "feature": FINAL_FEATURES,
    "mi_score": selector_all.scores_
}).sort_values("mi_score", ascending=False).reset_index(drop=True)

rank_df.insert(0, "rank", rank_df.index + 1)
rank_df



,rank,feature,mi_score
0,1,humidity,1.729954
1,2,moisture_index,1.669996
2,3,rainfall,1.637358
3,4,K,1.629636
4,5,P,1.308141
5,6,P_to_K,1.300008
6,7,temperature,1.017901
7,8,N,0.982875
8,9,ph,0.686067
9,10,ph_dev,0.348753




The ranked-feature output and selected-feature lists show that reducing to fewer than 10 features does not improve performance in our setup.  
Therefore, we keep all 10 features (7 original + 3 engineered) for the final modeling stage.


## 6.Final Feature Set for Modeling

Final predictors used in `03_modeling.ipynb`:

- `N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`
- `moisture_index`, `P_to_K`, `ph_dev`

These retain full baseline information while adding small, domain-justified complexity.

